<a href="https://colab.research.google.com/github/Damini-Walia/sms_text_classification/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS classification ("spam or ham")
*Solution by Damini Walia*


In [ ]:
# import libraries
'''
- The default import, doesn't load properly
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)
'''


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Inspect the files
! head -n 5 ./train-data.tsv

In [ ]:
! head -n 5 ./valid-data.tsv

In [ ]:
df_train = pd.read_csv(train_file_path, delimiter='\t', header=None, names=['class', 'message'])
df_test = pd.read_csv(test_file_path, delimiter='\t', header=None, names=['class', 'message'])

In [ ]:
df_train

In [ ]:
df_test

## Inspecting and cleaning the datasets

In [ ]:
# missing/invalid values in the training dataset.
invalid_values_train = df_train.isnull().sum(axis=1)
invalid_rows_train = invalid_values_train[invalid_values_train > 0]
num_invalid_rows_train = invalid_rows_train.shape[0]
num_invalid_rows_train

In [ ]:
# Make sure all values are either ham or spam in the training dataset
df_train['class'].unique()

In [ ]:
# Make sure all values are either ham or spam in the validation/testing dataset
df_test['class'].unique()

In [ ]:
# one-hot encoding the test/train datasets class columns
df_train['class'] = df_train['class'].replace(to_replace={'spam': 0, 'ham': 1})
df_test['class'] = df_test['class'].replace(to_replace={'spam': 0, 'ham': 1})

print('\n\ndf_train unique class values:', df_train['class'].unique())
print('df_test unique class values:', df_test['class'].unique())

In [ ]:
# Check data types
print('train datatypes:', df_train.dtypes)
print('test datatypes:', df_test.dtypes)

This is good - the `object` data type represents strings in pandas.

In [ ]:
# missing/invalid values in the testing dataset.
invalid_values_test = df_test.isnull().sum(axis=1)
invalid_rows_test = invalid_values_test[invalid_values_test > 0]
num_invalid_rows_test = invalid_rows_test.shape[0]
num_invalid_rows_test

In [ ]:
df_train.describe()

In [ ]:
df_test.describe()

In [ ]:
duplicates = df_train[df_train.duplicated()]
duplicates

In [ ]:
# Drop duplicates - why re-train on the same messages?
df_train = df_train.drop_duplicates()
df_train

In [ ]:
# Let's take a look at the distribution of lengths of the text messages
# Training messages
train_msg_lengths = df_train.message.apply(len)
print('Training messages:')
print('The longest message is', max(train_msg_lengths), 'characters')
print('Longest message:', max(df_train.message, key=len))
print('The shortest message:', min(df_train.message, key=len), '\n')
print(train_msg_lengths)

# Testing/validation messages
test_msg_lengths = df_test.message.apply(len)
print('\n\nValidation/Testing messages')
print('The longest message is', max(test_msg_lengths), 'characters')
print('Longest message:', max(df_test.message, key=len))
print('The shortest message:', min(df_test.message, key=len), '\n')
print(test_msg_lengths)

In [ ]:
plt.rcParams['figure.figsize'] = (12, 8)
plt.hist([train_msg_lengths, test_msg_lengths], bins=40)
plt.title('Histogram of Message Lengths in the training and testing datasets')
plt.xlabel('Length [characters]')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# create boxplot of the message lengths

plt.rcParams['figure.figsize'] = (12, 8)
msg_len_data = [train_msg_lengths, test_msg_lengths]
plt.boxplot(msg_len_data, tick_labels=['Training', 'Testing'])

# add title and axis labels
plt.title('Boxplot of Message lengths')
plt.xlabel('Messages')
plt.ylabel('Lengths')

# show the plot
plt.show()

How many spam messages vs. ham messages are there in the sets?

In [ ]:
train_class_count = df_train['class'].value_counts().values
test_class_count = df_test['class'].value_counts().values
print('train dataset class count:', train_class_count)
print('test dataset class count:', test_class_count)

In [ ]:
bar_width = 0.5
plt.bar([1,3],train_class_count, width=bar_width)
plt.bar([2,4],test_class_count, width=bar_width)
plt.xticks([1,2,3,4],['Train ham', 'Test ham', 'Train spam', 'Test spam'])
plt.show()

## Machine learning prediction with a RNN

Todo's:

*   [Check out tutorial on loading and pre-processing text in TensorFlow](https://www.tensorflow.org/tutorials/load_data/text) ✅
*   Figure out how to load the text into a text vectorization layer and vectorize the vocabulary ✅
* Build the model
* Train the model on the text with paired labels



In [ ]:
# Create the tensorflow training dataset
train_labels =  df_train["class"].values
train_ds = tf.data.Dataset.from_tensor_slices(
    (df_train["message"].values, train_labels)
)

# Print a couple of lines to see what it looks like
for i, e in enumerate(train_ds.as_numpy_iterator()):
  print(e)
  if i == 5:
    break

In [ ]:
# Create the tensorflow testing dataset
test_labels =  df_test["class"].values
test_ds = tf.data.Dataset.from_tensor_slices(
    (df_test["message"].values, test_labels)
)

# Print a couple of lines to see what it looks like
for i, e in enumerate(test_ds.as_numpy_iterator()):
  print(e)
  if i == 5:
    break

In [ ]:
print(len(df_train))

In [ ]:
BUFFER_SIZE = 1000
BATCH_SIZE = 128

### Preparing the data pipelines

Here is a breakdown of what each function in the pipeline does:

`shuffle(BUFFER_SIZE)`: This function shuffles the training data to avoid any biases that may be present in the original dataset. The BUFFER_SIZE parameter specifies the number of elements from the dataset to use as a buffer while shuffling. A larger buffer size leads to better shuffling, but also requires more memory.

`.batch(BATCH_SIZE)`: This function groups the shuffled data into batches of size BATCH_SIZE, which is a hyperparameter that determines the number of training examples processed in each training iteration. A larger batch size can speed up training, but also requires more memory.

`.prefetch(tf.data.AUTOTUNE)`: This function prefetches data from the next batch, allowing the GPU to work on the current batch while the CPU is loading the next one. The tf.data.AUTOTUNE parameter automatically tunes the prefetch buffer size based on available system resources, leading to optimal performance.

In [ ]:
train_ds = train_ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

### Encode/vectorize the text



In [ ]:
vec = tf.keras.layers.TextVectorization(
    output_mode='int',  # Vectorize the text as integers
    max_tokens=2000,  # Most important words should fit inside a 2K word vocab.
    output_sequence_length=500,  # Will fit most sentences, except a few outliers
)

# the map function only returns the text, we're not adapting the labels which
# are ignored by the function.
vec.adapt(train_ds.map(lambda text, label: text))

In [ ]:
vocab = np.array(vec.get_vocabulary())
vocab[:32]

Show an example of an encoded text message

In [ ]:
example_text = str(df_train.message[0])
example_text

In [ ]:
encoded_example = vec(example_text).numpy()[:35]
encoded_example

In [ ]:
vocab = np.array(vec.get_vocabulary())
print("Original: ", example_text)
print("Round-trip: ", " ".join(vocab[encoded_example]))

### Create and train the RNN model

This code is based on the [RNN text classification tutorial from Tensorflow.org](https://www.tensorflow.org/text/tutorials/text_classification_rnn)

![image](https://www.tensorflow.org/static/text/tutorials/images/bidirectional.png)

This code defines a TensorFlow Keras sequential model for a neural network with the following layers:



1.   The vectorization layer vec that we defined and created earlier
2.   An embedding layer with a vocabulary size equal to the length of the vocabulary of the vectorization layer, an embedding dimension of 64, and masking of zero values. Masking zero values means that the padding 0 values of the input are not considered for the training. The purpose of the embedding layer is to convert the indices of the vocabulary into sequences of vectors. The vectors are trainable and after training words with similar meanings often have similar vectors.
3.   A bidirectional LSTM layer with 64 units, which returns sequences that are used by the next layer. The bidirectionality allows text messages to be processed in both directions (see the tutorial)
4.   Another bidirectional LSTM layer with 32 units, which does not return sequences.
5.   A fully connected dense layer with 64 units and ReLU activation function.
6.   A dropout layer with a rate of 0.3.
7.   A final fully connected dense layer with 1 unit, which outputs the logits.


The model is compiled using the binary cross-entropy loss function for binary classification problems, the Adam optimizer with a learning rate of 1e-4, and accuracy as the metric to monitor during training.

In [ ]:
# Create the model
model = tf.keras.Sequential([
    vec,
    tf.keras.layers.Embedding(
        len(vec.get_vocabulary()),  # Input dim: size of vocab
        64,  # output dim: Dimension of the dense embedding
        mask_zero=True,
    ),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64,  return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1)
])


model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=['accuracy'],
)

This code fits the compiled model to the training data in `train_ds` for 10 epochs, with the validation data in `test_ds` used for validation at the end of each epoch. `validation_steps` is set to 30, which means that the model will evaluate the validation data for 30 steps (i.e., batches) at the end of each epoch.

During the training process, the model will update its parameters based on the binary cross-entropy loss function and the Adam optimizer. The validation accuracy will also be monitored during training. The `fit()` method returns a `history` object that contains information about the training and validation loss and accuracy at each epoch. This object can be used to visualize the training and validation curves and to diagnose potential issues such as overfitting.

In [ ]:
val_steps = len(df_test)/BATCH_SIZE
val_steps

Fit the model - this takes forever (~30-40 minutes). Definitely want to use (T/G)PUs here

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    validation_steps=int(val_steps),
    epochs=16,
)

### Results from 16 epochs


In [ ]:
for i, e in enumerate(history.history['val_accuracy']):
  print(f'validation accuracy for epoch {i+1}: {round(100*e, 2)} %')

In [ ]:
for i, e in enumerate(history.history['loss']):
  print(f'loss for epoch {i+1}: {round(e, 3)}')

Let's see how our model performed

In [ ]:
def plot_graphs(h, metric):
    plt.plot(h.history[metric])
    plt.plot(h.history['val_'+metric])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend([metric, 'val_'+metric])

In [ ]:
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plot_graphs(history, 'accuracy')
plt.ylim(None, 1)
plt.subplot(1, 2, 2)
plot_graphs(history, 'loss')
plt.ylim(0, None)

# FreeCodeCamp Certification test part

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    ps = model.predict(tf.constant([pred_text]))
    print(ps)
    p = ps[0][0]
    result = [p, "ham" if p > 0.5 else "spam"]
    print(result)
    return [p, "ham" if p > 0.5 else "spam"]


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
